[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Why Gradient Boosting &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' tables and model, rebuilt here from the same made-up stations so that each answer
runs on its own.

In [2]:
def as_numbers(rows):
    return pd.get_dummies(rows[FEATURES], columns=["terrain"], dtype=float)

X_train, X_test = as_numbers(train), as_numbers(test)
y_train, y_test = train["tmin"], test["tmin"]


def boost_by_hand(learning_rate, depth, rounds, report):
    prediction_train = np.full(len(y_train), y_train.mean())
    prediction_test = np.full(len(y_test), y_train.mean())
    for round_number in range(1, rounds + 1):
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_train, y_train - prediction_train)
        prediction_train += learning_rate * tree.predict(X_train)
        prediction_test += learning_rate * tree.predict(X_test)
        if round_number in report:
            print(round_number, "test", round(mean_squared_error(y_test, prediction_test), 3))

**1.** Run the hand-written loop again with a learning rate of 0.3, and print the test loss after 10, 50 and 300 rounds to see whether a larger step helps.

In [3]:
boost_by_hand(learning_rate=0.3, depth=3, rounds=300, report=(10, 50, 300))

10 test 5.671
50 test 2.936
300 test 2.53


At 0.3 the test loss is 5.671 after 10 rounds, 2.936 after 50 and 2.53 after 300, against 14.572, 3.941 and 2.61 at 0.1. The larger step gets there sooner, and in this run its 300 rounds also end slightly lower. The **Learning Rate and Trees** notebook shows the trade between the step and the number of rounds.

**2.** Run the loop with trees of depth 1 in place of 3, for 300 rounds, and print the test loss.

In [4]:
boost_by_hand(learning_rate=0.1, depth=1, rounds=300, report=(300,))

300 test 4.499


Trees of depth 1 reach only 4.499 after 300 rounds. A tree with one split looks at one feature, so a sum of them adds up each feature's effect on its own and cannot describe a combination such as a clear, calm night, the same limit the linear model has.

**3.** Train LightGBM with 50 trees and with 1,000 trees, at a learning rate of 0.1, and print the test loss of each beside the 300-tree model's.

In [5]:
for trees in (50, 300, 1000):
    model = lgb.LGBMRegressor(n_estimators=trees, learning_rate=0.1, random_state=0, verbose=-1)
    model.fit(train[FEATURES], y_train)
    print(trees, "trees:", round(mean_squared_error(y_test, model.predict(test[FEATURES])), 3))

50 trees: 2.758
300 trees: 2.542
1000 trees: 2.654


50 trees leave 2.758, 300 trees 2.542 and 1,000 trees 2.654, so more trees stopped helping somewhere between 300 and 1,000, and the test stations cannot be used to find where without spoiling them as a test. The **Early Stopping** notebook lets the validation stations decide.

**4.** Train an XGBoost model with `max_depth=3`, the depth of the hand-written loop's trees, for 300 rounds at 0.1, and print its test loss.

In [6]:
shallow = xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0)
shallow.fit(train[FEATURES], y_train)
print("XGBoost, depth 3:", round(mean_squared_error(y_test, shallow.predict(test[FEATURES])), 3))

XGBoost, depth 3: 2.646


The test loss is 2.646, close to the 2.61 of the hand-written loop with the same depth, rate and number of trees. XGBoost's trees still differ from the loop's in details such as a penalty on the size of a leaf's value, which the **Regularization and Sampling** notebook covers.

**5.** Make a one-row table for a valley station at latitude 69.0, 250 meters up and 60 kilometers from the coast, on day 15 with a cloud cover of 0, a wind of 0.5 meters a second and 40 centimeters of snow, and print what `lightgbm_model` predicts for it. Give `terrain` the same categories as the training table.

In [7]:
lightgbm_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_model.fit(train[FEATURES], y_train)

night = pd.DataFrame({
    "latitude": [69.0], "elevation_m": [250], "coast_km": [60.0],
    "terrain": pd.Categorical(["valley"], categories=TERRAINS),
    "day_of_year": [15], "cloud_cover": [0.0], "wind_ms": [0.5], "snow_cm": [40.0],
})
print(round(float(lightgbm_model.predict(night[FEATURES])[0]), 1))

-21.9


The model predicts −21.9 degrees. A clear, calm January night in a valley with snow on the ground combines every condition that makes these made-up stations cold.

---

&#8592; **Back to:** [Why Gradient Boosting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/01-why-gradient-boosting.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
